In [1]:
import math
import numpy as np
import torch
import torch.nn as nn
import os
from ultralytics.models.yolo.detect import DetectionTrainer 

# ================= 1. 定义全流程防护隐私引擎 (Full Protection Engine) =================
class PrivacyEngine_Full_Protection:
    def __init__(self, model, epsilon=1.0, strategy='uniform', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        self.total_epochs = epochs
        
        # 识别 Head 层
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))
        
        # ========== Warm-up 阶段参数 (Phase I: Weak Privacy) ==========
        # 建议设置：
        # clip_val=5.0: 比 2.0 宽容一点，防止迁移初期梯度过大导致“硬着陆”
        # sigma=0.5:  轻量噪声，足以破坏 DLG 像素级还原，又不至于毁掉特征
        self.warmup_clip_val = 5.0  
        self.warmup_sigma_base = 0.5 

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        # 仅用于 Phase II (Epoch >= 3)
        min_decay = 0.1
        progress = current_epoch / self.total_epochs
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        # 收集梯度
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        if not param_list: return
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        current_device = next(self.model.parameters()).device

        # ================= Phase I: Warm-up 轻量防护 (Epoch 0-2) =================
        if current_epoch < 3:
            # 1. 监控真实梯度范数 (Monitor Real Gradient Norm)
            # 这步非常关键！用来验证 5.0 的阈值是否合适
            grad_norms = [p.grad.norm(2).item() for p in param_list]
            real_median = np.median(grad_norms)
            
            # 2. 打印日志 (防止刷屏，每轮只打一次)
            if not hasattr(self, '_warmup_logged_epoch') or self._warmup_logged_epoch != current_epoch:
                print(f"🛡️ [Phase I: Warm-up] Epoch {current_epoch}")
                print(f"   📊 真实梯度中位数: {real_median:.4f} | 设定裁剪阈值: {self.warmup_clip_val}")
                print(f"   🔒 注入轻量噪声 (Sigma): {self.warmup_sigma_base}")
                if real_median > self.warmup_clip_val * 2:
                    print(f"   ⚠️ 警告: 真实梯度远大于裁剪阈值，模型正在经历强约束！")
                self._warmup_logged_epoch = current_epoch

            # 3. 执行轻量保护
            for p in param_list:
                # 固定阈值裁剪
                torch.nn.utils.clip_grad_norm_(p, self.warmup_clip_val)
                # 固定噪声干扰
                p.grad.add_(torch.randn_like(p.grad) * self.warmup_sigma_base)
            
            return # Warm-up 阶段结束，直接返回

        # ================= Phase II: ST-ADM 强力防护 (Epoch 3-29) =================
        
        # 1. 计算退火系数
        decay_factor = self._get_noise_multiplier(current_epoch)

        # 2. AGC 自适应裁剪
        grad_norms = [p.grad.norm(2).item() for p in param_list]
        current_median = np.median(grad_norms)
        # AGC 核心公式
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 3. 打印日志
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"⚔️ [Phase II: ST-ADM] Epoch {current_epoch} | AGC Clip: {clip_val:.4f} | Decay: {decay_factor:.4f}")
            self._logged_this_epoch = current_epoch

        # 4. 策略权重计算
        factors = [1.0 for _ in names_list]
        if self.strategy == 'adaptive_smart':
            for i, n in enumerate(names_list):
                try:
                    layer_idx = int(n.split('.')[1])
                    if layer_idx in self.head_indices: factors[i] = 1.5
                    elif layer_idx in self.backbone_indices: factors[i] = 0.8
                except: pass
        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors)

        # 5. 执行强力保护
        for idx, p in enumerate(param_list):
            layer_eps = self.epsilon * weights[idx]
            
            # 裁剪
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            # 加噪
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)


# ================= 2. 定义训练器 =================
class Trainer_Full_Protection(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 调用全流程防护引擎
        self.privacy_engine = PrivacyEngine_Full_Protection(
            model, epsilon=1.0, strategy='uniform', epochs=30
        )
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        # 兜底裁剪，防止数值溢出
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 20: Full Process Protection =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
# 使用 GC10 预训练权重
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始 Exp 20: 全流程隐私防护实验 (Warm-up Protected + ST-ADM)")
print("📝 配置: 前3轮轻量DP (Clip=5.0, Sigma=0.5) -> 后续 ST-ADM (AGC+Annealing)")
print("🎯 目的: 消除 Warm-up 阶段的泄露风险，实现全生命周期防护")

if os.path.exists(PRETRAINED_GC10):
    trainer_full = Trainer_Full_Protection(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_exp1', # 或者是 result_ablation，看你想存哪
        'name': '20_Full_Protection_Warmup',
        'device': '0',
        'exist_ok': True,
        'freeze': 0
    })
    trainer_full.train()
else:
    print(f"❌ 找不到预训练权重: {PRETRAINED_GC10}")

🚀 开始 Exp 20: 全流程隐私防护实验 (Warm-up Protected + ST-ADM)
📝 配置: 前3轮轻量DP (Clip=5.0, Sigma=0.5) -> 后续 ST-ADM (AGC+Annealing)
🎯 目的: 消除 Warm-up 阶段的泄露风险，实现全生命周期防护
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/re

In [2]:
import math
import numpy as np
import torch
import torch.nn as nn
import os
from ultralytics.models.yolo.detect import DetectionTrainer 

class PrivacyEngine_Sparsification:
    def __init__(self, model, epsilon=1.0, strategy='uniform', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        self.total_epochs = epochs
        
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))
        
        # ========== Exp 21 新策略参数 ==========
        # 1. 稀疏化率: 0.5 表示把 50% 的小梯度置为 0
        self.warmup_prune_rate = 0.5  
        # 2. 宽松裁剪: 允许大梯度通过，保证迁移学习的方向
        self.warmup_loose_clip = 50.0 
        # 3. 微量噪声: 仅仅为了破坏数值确定性
        self.warmup_tiny_sigma = 0.01

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        min_decay = 0.1
        progress = current_epoch / self.total_epochs
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        if not param_list: return
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        current_device = next(self.model.parameters()).device

        # ================= Phase I: Warm-up 稀疏化防御 (Epoch 0-2) =================
        if current_epoch < 3:
            # 1. 监控
            if not hasattr(self, '_warmup_logged_epoch') or self._warmup_logged_epoch != current_epoch:
                grad_norms = [p.grad.norm(2).item() for p in param_list]
                real_median = np.median(grad_norms)
                print(f"🛡️ [Warm-up Sparsification] Epoch {current_epoch}")
                print(f"   📊 真实梯度中位数: {real_median:.4f} | 宽松裁剪: {self.warmup_loose_clip}")
                print(f"   ✂️ 稀疏化率: {self.warmup_prune_rate*100}% (只保留大梯度)")
                self._warmup_logged_epoch = current_epoch

            for p in param_list:
                # A. 宽松裁剪 (防止极大值溢出，但不改变主要方向)
                torch.nn.utils.clip_grad_norm_(p, self.warmup_loose_clip)
                
                # B. 梯度稀疏化 (核心防御)
                g_flat = p.grad.abs().view(-1)
                k = int(self.warmup_prune_rate * g_flat.numel())
                if k > 0:
                    # 找到第 k 小的值作为阈值
                    threshold_val = torch.kthvalue(g_flat, k).values
                    # 生成掩码：小于阈值的置 0
                    mask = (p.grad.abs() >= threshold_val).float()
                    p.grad.mul_(mask)
                
                # C. 注入微量噪声 (防止精确匹配)
                p.grad.add_(torch.randn_like(p.grad) * self.warmup_tiny_sigma)
            
            return # Warm-up 结束

        # ================= Phase II: ST-ADM (Epoch 3-29) =================
        # (保持原有的 Exp 18 逻辑不变)
        decay_factor = self._get_noise_multiplier(current_epoch)
        grad_norms = [p.grad.norm(2).item() for p in param_list]
        current_median = np.median(grad_norms)
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"⚔️ [Phase II: ST-ADM] Epoch {current_epoch} | AGC Clip: {clip_val:.4f} | Decay: {decay_factor:.4f}")
            self._logged_this_epoch = current_epoch

        factors = [1.0 for _ in names_list]
        # (Uniform 策略省略，直接写核心逻辑)
        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors)

        for idx, p in enumerate(param_list):
            layer_eps = self.epsilon * weights[idx]
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)

# ================= Trainer & Run =================
class Trainer_Sparsify(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        self.privacy_engine = PrivacyEngine_Sparsification(
            model, epsilon=1.0, strategy='uniform', epochs=30
        )
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始 Exp 21: 稀疏化防御实验")
print("🎯 策略: Warm-up 阶段使用 '梯度稀疏化' + '宽松裁剪'，彻底解决 DLG 风险")

if os.path.exists(PRETRAINED_GC10):
    trainer_sparse = Trainer_Sparsify(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_exp1',
        'name': '21_Sparsification_Warmup',
        'device': '0',
        'exist_ok': True,
        'freeze': 0
    })
    trainer_sparse.train()

🚀 开始 Exp 21: 稀疏化防御实验
🎯 策略: Warm-up 阶段使用 '梯度稀疏化' + '宽松裁剪'，彻底解决 DLG 风险
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.937, mosaic=1.0, multi_

/tmp/ipykernel_1365/3106116480.py:70: UserWarning: kthvalue CUDA does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:71.)
  threshold_val = torch.kthvalue(g_flat, k).values


       1/30      2.32G      1.795      3.367      1.918         78        640: 100% ━━━━━━━━━━━━ 90/90 6.5it/s 13.8s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 9.5it/s 0.6s0.2s
                   all        180        391      0.246      0.341      0.235     0.0888

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
🛡️ [Warm-up Sparsification] Epoch 1
   📊 真实梯度中位数: 1.0360 | 宽松裁剪: 50.0
   ✂️ 稀疏化率: 50.0% (只保留大梯度)
       2/30      2.45G      1.562      2.926      1.846         79        640: 0% ──────────── 0/90  0.1s

/tmp/ipykernel_1365/3106116480.py:70: UserWarning: kthvalue CUDA does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:71.)
  threshold_val = torch.kthvalue(g_flat, k).values


       2/30      2.45G      1.556      2.374       1.72         67        640: 100% ━━━━━━━━━━━━ 90/90 9.8it/s 9.1s<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 11.1it/s 0.5s.1s
                   all        180        391      0.382      0.549      0.461        0.2

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      2.45G      1.585      2.044      1.713         91        640: 1% ──────────── 1/90 2.6it/s 0.1s<34.7s🛡️ [Warm-up Sparsification] Epoch 2
   📊 真实梯度中位数: 1.3440 | 宽松裁剪: 50.0
   ✂️ 稀疏化率: 50.0% (只保留大梯度)
       3/30      2.45G      1.528      2.031      1.673         72        640: 2% ──────────── 2/90 3.5it/s 0.3s<25.0s

/tmp/ipykernel_1365/3106116480.py:70: UserWarning: kthvalue CUDA does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:71.)
  threshold_val = torch.kthvalue(g_flat, k).values


       3/30      2.45G      1.549      2.041      1.674         89        640: 100% ━━━━━━━━━━━━ 90/90 10.9it/s 8.3s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 11.2it/s 0.5s.4s
                   all        180        391      0.522      0.507      0.531      0.263

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      2.45G       1.37      1.811      1.528         79        640: 1% ──────────── 1/90 2.5it/s 0.1s<35.4s⚔️ [Phase II: ST-ADM] Epoch 3 | AGC Clip: 1.9915 | Decay: 0.9780
       4/30      2.45G      1.796      2.332      1.853         68        640: 100% ━━━━━━━━━━━━ 90/90 12.0it/s 7.5s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 12.0it/s 0.5s.3s
                   all        180        391      0.256      0.306      0.231     0.0811

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  In